In [22]:
import json
import os
import numpy as np
import pandas as pd
from thefuzz import fuzz, process

# ==========================================
# 0. FILE PATH CONFIGURATION & LOADING
# ==========================================
RAW_DIR = os.path.join("DATA", "RAW")

noida_path = os.path.join(RAW_DIR, "noida_clients.csv")
mohali_path = os.path.join(RAW_DIR, "mohali_accounts.csv")

print("Loading raw datasets...")
df_noida = pd.read_csv(noida_path)
df_mohali = pd.read_csv(mohali_path)

# ==========================================
# 1. NOIDA CLEANING & STRUCTURAL UNPACKING
# ==========================================
print("\n--- Cleaning Noida Dataset ---")

# A. Unpack nested JSON in 'client_details'
df_noida["client_details"] = df_noida["client_details"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
df_noida["City"] = df_noida["client_details"].apply(
    lambda x: x.get("city") if isinstance(x, dict) else np.nan
)
df_noida["Zip"] = df_noida["client_details"].apply(
    lambda x: x.get("zip") if isinstance(x, dict) else np.nan
)
df_noida = df_noida.drop(columns=["client_details"])

# B. Extract Email and Phone using Regular Expressions
email_pattern = r"([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)"
phone_pattern = r"(\+91-\d{10})"

df_noida["Email"] = df_noida["contact_notes"].str.extract(email_pattern)
df_noida["Phone"] = df_noida["contact_notes"].str.extract(phone_pattern)
df_noida = df_noida.drop(columns=["contact_notes"])

# C. Median Imputation for missing sales
noida_sales_median = df_noida["monthly_sales_inr"].median()
df_noida["monthly_sales_inr"] = df_noida["monthly_sales_inr"].fillna(
    noida_sales_median
)

# D. Currency Conversion (INR to USD, 83.0 conversion rate)
df_noida["Revenue_USD"] = (df_noida["monthly_sales_inr"] / 83.0).round(2)
df_noida["Revenue_INR"] = df_noida["monthly_sales_inr"].round(2)

# E. Impute text nulls & add missing schema dimensions
df_noida["City"] = df_noida["City"].fillna("Unknown")
df_noida["Zip"] = (
    df_noida["Zip"].replace("N/A", "Unknown").fillna("Unknown")
)
df_noida["Email"] = df_noida["Email"].fillna("Unknown")
df_noida["Phone"] = df_noida["Phone"].fillna("Unknown")
df_noida["Region"] = "Noida"
df_noida["Company"] = "Not Disclosed"
df_noida["Age_of_Account_Years"] = np.nan

df_noida_clean = df_noida.rename(columns={"client_id": "Account_ID"})

print(f"Noida records processed: {len(df_noida_clean):,}")

# ==========================================
# 2. MOHALI CLEANING & ANOMALY RESOLUTION
# ==========================================
print("\n--- Cleaning Mohali Dataset ---")

# A. Categorical Text Standardization via Fuzzy Matching & Dictionary Mapping
target_companies = ["HCLTech", "TCS", "Infosys", "Wipro"]
alias_map = {
    "Infy": "Infosys",
    "Tata Consultancy Services": "TCS",
    "Tata Consult": "TCS",
    "tcs ltd": "TCS",
    "hcl": "HCLTech",
    "HCL Technologies": "HCLTech",
    "hcl tech ltd": "HCLTech",
}


def standardize_company(name):
    clean_str = str(name).strip()
    if clean_str in alias_map:
        return alias_map[clean_str]

    best_match = process.extractOne(
        clean_str, target_companies, scorer=fuzz.token_set_ratio
    )
    if best_match and best_match[1] >= 75:
        return best_match[0]
    return clean_str


df_mohali["Clean_Company"] = df_mohali["Company"].apply(standardize_company)

# B. Median Imputation for missing revenue
mohali_rev_median = df_mohali["Revenue_USD"].median()
df_mohali["Revenue_USD"] = df_mohali["Revenue_USD"].fillna(mohali_rev_median)

# C. Capping extreme Revenue Outliers (currency multiplier typos)
q1_rev = df_mohali["Revenue_USD"].quantile(0.25)
q3_rev = df_mohali["Revenue_USD"].quantile(0.75)
iqr_rev = q3_rev - q1_rev
upper_rev_bound = q3_rev + 3.0 * iqr_rev

df_mohali["Revenue_USD"] = np.where(
    df_mohali["Revenue_USD"] > upper_rev_bound,
    upper_rev_bound,
    df_mohali["Revenue_USD"],
).round(2)

# D. Capping impossible Account Age Outliers (e.g., 200-500 years)
q1_age = df_mohali["Age_of_Account_Years"].quantile(0.25)
q3_age = df_mohali["Age_of_Account_Years"].quantile(0.75)
iqr_age = q3_age - q1_age
upper_age_bound = q3_age + 1.5 * iqr_age

df_mohali["Age_of_Account_Years"] = np.where(
    df_mohali["Age_of_Account_Years"] > upper_age_bound,
    upper_age_bound,
    df_mohali["Age_of_Account_Years"],
).round(1)

# E. Schema Alignment & Dropping Raw Messy Columns (Avoids Duplicate Column Error)
df_mohali_clean = df_mohali.drop(columns=["Company"]).copy()
df_mohali_clean = df_mohali_clean.rename(
    columns={"AccountID": "Account_ID", "Clean_Company": "Company"}
)

df_mohali_clean["Region"] = "Mohali"
df_mohali_clean["Revenue_INR"] = (
    df_mohali_clean["Revenue_USD"] * 83.0
).round(2)
df_mohali_clean["City"] = "Mohali"
df_mohali_clean["Zip"] = "Unknown"
df_mohali_clean["Email"] = "Unknown"
df_mohali_clean["Phone"] = "Unknown"

print(f"Mohali records processed: {len(df_mohali_clean):,}")

# ==========================================
# 3. SCHEMA HARMONIZATION & VERTICAL MERGE
# ==========================================
print("\n--- Harmonizing and Merging Datasets ---")

master_columns = [
    "Account_ID",
    "Region",
    "Company",
    "Revenue_USD",
    "Revenue_INR",
    "Age_of_Account_Years",
    "City",
    "Zip",
    "Email",
    "Phone",
]

# Ensure uniqueness of columns before slicing
df_noida_clean = df_noida_clean.loc[:, ~df_noida_clean.columns.duplicated()]
df_mohali_clean = df_mohali_clean.loc[
    :, ~df_mohali_clean.columns.duplicated()
]

noida_final = df_noida_clean[master_columns]
mohali_final = df_mohali_clean[master_columns]

# Concatenate vertically into the final master dataset
combined_master_df = pd.concat(
    [noida_final, mohali_final], ignore_index=True
)

# ==========================================
# 4. EXPORT OUTPUTS TO CSV
# ==========================================
print("\n--- Exporting Cleaned CSV Files ---")

noida_final.to_csv("noida_cleaned.csv", index=False)
mohali_final.to_csv("mohali_cleaned.csv", index=False)
combined_master_df.to_csv("harmonized_master_companies.csv", index=False)

print("Export complete:")
print(f"  • noida_cleaned.csv               : {len(noida_final):,} rows")
print(f"  • mohali_cleaned.csv              : {len(mohali_final):,} rows")
print(
    f"  • harmonized_master_companies.csv : {len(combined_master_df):,} rows"
)
print(f"Combined Shape: {combined_master_df.shape}")

Loading raw datasets...

--- Cleaning Noida Dataset ---
Noida records processed: 100,000

--- Cleaning Mohali Dataset ---
Mohali records processed: 250,000

--- Harmonizing and Merging Datasets ---

--- Exporting Cleaned CSV Files ---
Export complete:
  • noida_cleaned.csv               : 100,000 rows
  • mohali_cleaned.csv              : 250,000 rows
  • harmonized_master_companies.csv : 350,000 rows
Combined Shape: (350000, 10)
